***

Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
pd.options.display.float_format = '{:.1f}'.format

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    # SharePoint
    path_out  = os.path.join(path_users
                             , 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents'
                             , 'Process Revamp'
                             , 'Task 9. Collect new data'
                             , 'Census')

path_code    = os.path.join(path_git, 'Data', 'EIA')
path_config0 = os.path.join(path_git , 'config')
path_config  = os.path.join(path_code, 'config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://www.eia.gov/opendata/documentation.php
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

***

Importing

***

In [ ]:

root_ = 'https://api.eia.gov/v2'
api_key_ = f'/?api_key={api_key}'

query = f"{root_}{api_key_}"
query

# Use requests package to call out to the API
response = requests.get(query).text
response = response.replace('null', '"null"')
response = ast.literal_eval(response)

response

In [ ]:
root_ = 'https://api.eia.gov/v2'
api_key_ = f'/?api_key={api_key}'

query = f"{root_}{api_key_}"
query

# Use requests package to call out to the API
response = requests.get(query).text
response = response.replace('null', '"null"')
response = ast.literal_eval(response)


list_df = []

for id in range(len(response['response']['routes'])):
    list_df.append(pd.DataFrame(response['response']['routes'][id], index = [0]))


df_cats = pd.concat(list_df)
df_cats = df_cats.reset_index(drop = True)

df_cats.columns = ['categories', 'category_name', 'category_description']

df_cats

In [ ]:
list(df_cats['categories'].unique())

In [ ]:
categories = list(df_cats['categories'].unique())
categories.remove('ieo')

categories

In [ ]:

# # %%capture cap# --no-stderr


# root_ = 'https://api.eia.gov/v2'
# api_key_ = f'/?api_key={api_key}'


# for cat in categories:
    
#     cat_ = f'/{cat}'
    
#     query = f"{root_}{cat_}{api_key_}"
    
#     # Use requests package to call out to the API
#     response = requests.get(query).text
#     response = response.replace('null', '"null"')
#     response = ast.literal_eval(response)


#     print(f'Category: {cat}'  );print('')
#     print('')

#     print(f'Query: {query}');print('')

#     print('Keys in response: ')
#     print(list(response['response'].keys()))
#     print('')

#     print('Full response:')
#     print(response['response'])
#     print('')
#     print('')
#     print('')

    
# # with open(os.path.join(path_code, 'API Structure.txt'), 'w') as f:
# #     f.write(cap.stdout)
    

In [ ]:
categories = list(df_cats['categories'].unique())
categories

In [ ]:
root_ = 'https://api.eia.gov/v2'
api_key_ = f'/?api_key={api_key}'


has_routes = ['coal', 'electricity', 'natural-gas', 'nuclear-outages', 'petroleum', 'densified-biomass', 'aeo', 'co2-emissions']
categories_to_keep = ['coal', 'electricity', 'natural-gas', 'nuclear-outages', 'petroleum', 'densified-biomass', 'co2-emissions']

list_df_cat = []

for cat in categories_to_keep:

    cat_ = f'/{cat}'
    
    query = f"{root_}{cat_}{api_key_}"
    
    # Use requests package to call out to the API
    response = requests.get(query).text
    response = response.replace('null', '"null"')
    response = ast.literal_eval(response)
    
    
    list_df_id = []

    if cat in has_routes:
        for id in range(len(response['response']['routes'])):
            df_routes = pd.DataFrame(response['response']['routes'][id], index = [0])
            df_routes = df_routes.rename(columns = {'id':'route', 'name':'route_name'})
            list_df_id.append(df_routes)
    
    
    df_routes = pd.concat(list_df_id)
    df_routes = df_routes.reset_index(drop = True)
    
    
    df_routes['category'] = cat
    list_df_cat.append(df_routes)


df_routes = pd.concat(list_df_cat)
df_routes = df_routes.reset_index(drop = True)
df_routes = df_routes.set_index('category').reset_index()
df_routes['route_name'] = df_routes['route_name'].str.replace('\\/', '/')
df_routes['route_name'] = df_routes['route_name'].str.replace('\\' , '/')
df_routes['route_name'] = df_routes['route_name'].str.replace(' / ', '/')

display(df_routes)
# df_routes.to_excel(os.path.join(path_config, 'Routes.xlsx'), index=False)

***

With a route

***

In [ ]:
category = 'electricity'
cat_ = f'/{category}'

route = 'retail-sales'
route_ = f'/{route}'

metric = 'price'
# data_ = f'/data/?data[]={metric}'
data_ = f'/data'


# api_key_ = f'&api_key={api_key}'
api_key_ = f'?api_key={api_key}'


query = f"{root_}{cat_}{route_}{data_}{api_key_}"
print(query); print('')

# Use requests package to call out to the API
response = requests.get(query).text
response = response.replace('null', '"null"')
response = ast.literal_eval(response)

response['response']

In [ ]:

list_df = []

for row in range(len(response['response']['data'])):
    list_df.append(pd.DataFrame(response['response']['data'][row], index = [0]))

df_test1 = pd.concat(list_df)
df_test1 = df_test1.reset_index(drop = True)
df_test1 = df_test1.sort_values('period')

df_test1

***

With a frequency

***

In [ ]:
category = 'total-energy'
cat_ = f'/{category}'

freq = 'annual'
data_ = f'/data/?frequency={freq}'

api_key_ = f'&api_key={api_key}'

query = f"{root_}{cat_}{data_}{api_key_}"
print(query); print('')

# Use requests package to call out to the API
response = requests.get(query).text
response = response.replace('null', '"null"')
response = ast.literal_eval(response)

response['response']

In [ ]:
list_df = []

for row in range(len(response['response']['data'])):
    list_df.append(pd.DataFrame(response['response']['data'][row], index = [0]))

df_test2 = pd.concat(list_df)
df_test2 = df_test2.sort_values('period')
df_test2 = df_test2.reset_index(drop = True)


df_test2

***

Testing

***

Petroleum

In [ ]:
params = {
    'api_key': api_key,
    "frequency": "annual",
    "data[0]": 'value'
    # 'facets[stateid][]': 'CA',
    # 'facets[sectorid][]': 'RES'
}

In [ ]:
root_ = 'https://api.eia.gov/v2'

cat = 'petroleum'
cat_ = f'/{cat}'

# route = 'pri/gnd'
route = 'pri'
route_ = f'/{route}'

# freq = 'annual'
# value = 'value'
# geography = 'SCA'
# data_ = f'/data/?frequency={freq}&data[0]={value}&facets[duoarea][]={geography}'
data_ = f'/data'


# api_key_ = f'&api_key={api_key}'
api_key_ = f'/?api_key={api_key}'

# query = f"{root_}{cat_}{route_}{data_}{api_key_}"
# url = f"{root_}{cat_}{route_}{data_}{api_key_}"
url = f"{root_}{cat_}{route_}{data_}{api_key_}"



# # Use requests package to call out to the API
# response = requests.get(query).text
# response = response.replace('null', '"null"')
# response = ast.literal_eval(response)


# Use requests package to call out to the API
response = requests.get(url, params=params)
response = response.json()

response


In [ ]:
response['response'][]

In [ ]:
list_df = []

for row in range(len(response['response']['data'])):
    list_df.append(pd.DataFrame(response['response']['data'][row], index = [0]))

df_test3 = pd.concat(list_df)
df_test3 = df_test3.sort_values('period')
df_test3 = df_test3.reset_index(drop = True)

df_test3

# list_df = []

# for row in range(len(response['response']['data'])):
#     list_df.append(pd.DataFrame(response['response']['data'][row], index = [0]))

# df_test3 = pd.concat(list_df)
# df_test3 = df_test3.sort_values('period')
# df_test3 = df_test3.reset_index(drop = True)

# df_test3

In [ ]:
len(df_test3['series-description'].unique())

In [ ]:
root_ = 'https://api.eia.gov/v2'

cat = 'electricity'
cat_ = f'/{cat}'

route = 'retail-sales'
route_ = f'/{route}'

freq = 'annual'
value = 'price'
geography = 'CA'

data_ = f'/data/?frequency={freq}&data[0]={value}&facets[stateid][]={geography}'

api_key_ = f'&api_key={api_key}'

    
query = f"{root_}{cat_}{route_}{data_}{api_key_}"

# Use requests package to call out to the API
response = requests.get(query).text
response = response.replace('null', '"null"')
response = ast.literal_eval(response)

response

In [ ]:
list_df = []

for row in range(len(response['response']['data'])):
    list_df.append(pd.DataFrame(response['response']['data'][row], index = [0]))

df_test4 = pd.concat(list_df)
df_test4 = df_test4.sort_values('period')
df_test4 = df_test4.reset_index(drop = True)

df_test4

In [ ]:
params = {
    'api_key': api_key,
    "frequency": "annual",
    "data[0]": 'price',
    'facets[stateid][]': 'CA',
    'facets[sectorid][]': 'RES'
    }

In [ ]:
root_ = 'https://api.eia.gov/v2'

cat = 'electricity'
cat_ = f'/{cat}'

route = 'retail-sales'
route_ = f'/{route}'

data_ = f'/data'

# api_key_ = f'/?api_key={api_key}'


url = f"{root_}{cat_}{route_}{data_}"
# query = f"{root_}{cat_}{route_}{data_}{api_key_}"

print(url)
print('')

# Use requests package to call out to the API
response = requests.get(url, params=params)
response = response.json()

response

In [ ]:
list_df = []

for row in range(len(response['response']['data'])):
    list_df.append(pd.DataFrame(response['response']['data'][row], index = [0]))

df_test5 = pd.concat(list_df)
df_test5 = df_test5.sort_values('period')
df_test5 = df_test5.reset_index(drop = True)

df_test5

In [ ]:
root_ = 'https://api.eia.gov/v2'

cat = 'petroleum'
cat_ = f'/{cat}'

# route = 'pri/gnd'
route = 'pri'
route_ = f'/{route}'


api_key_ = f'/?api_key={api_key}'


query = f"{root_}{cat_}{route_}{api_key_}"

print(query)
print('')

# Use requests package to call out to the API
response = requests.get(query).text
response = response.replace('null', '"null"')
response = ast.literal_eval(response)


response
